# 01 — 探索性資料分析（EDA）

NBA 對戰勝率預測。目的：了解資料規模、標籤分布、缺值狀況與基本樣態，
為特徵工程與建模做準備。

> 執行前請先下載資料：`python -m src.download_nba_api`

In [1]:
import sys
from pathlib import Path

# 讓 notebook 能 import 專案的 src 套件
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data_loader, features, config

sns.set_theme(style='whitegrid')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

## 1. 載入資料

In [2]:
games = data_loader.load_games()
teams = data_loader.load_teams()
print(f'比賽數：{len(games)}')
print(f'賽季範圍：{games.SEASON.min()}–{games.SEASON.max()}')
print(f'日期範圍：{games.GAME_DATE_EST.min().date()} ~ {games.GAME_DATE_EST.max().date()}')
games.head()

比賽數：7059
賽季範圍：2018–2023
日期範圍：2018-10-16 ~ 2024-04-14


,GAME_ID,GAME_DATE_EST,SEASON,HOME_TEAM_ID,VISITOR_TEAM_ID,PTS_home,PTS_away,FG_PCT_home,FG_PCT_away,FT_PCT_home,FT_PCT_away,FG3_PCT_home,FG3_PCT_away,AST_home,AST_away,REB_home,REB_away,HOME_TEAM_WINS
0,21800002,2018-10-16,2018,1610612744,1610612760,108,100,0.442,0.363,0.944,0.649,0.269,0.270,28,21,58,45,1
1,21800001,2018-10-16,2018,1610612738,1610612755,105,87,0.433,0.391,0.714,0.609,0.297,0.192,21,18,55,47,1
2,21800009,2018-10-17,2018,1610612745,1610612740,112,131,0.424,0.531,0.750,0.773,0.333,0.400,21,36,37,54,0
3,21800012,2018-10-17,2018,1610612746,1610612743,98,107,0.398,0.379,0.833,0.786,0.286,0.333,21,20,47,56,0
4,21800013,2018-10-17,2018,1610612756,1610612742,121,100,0.543,0.432,0.875,0.700,0.559,0.303,35,28,44,38,1


## 2. 標籤分布：主隊勝率

In [3]:
dist = data_loader.label_distribution(games)
print(dist.rename({0: '客隊勝', 1: '主隊勝'}))

ax = dist.rename({0: 'Away win', 1: 'Home win'}).plot(kind='bar', color=['#c44', '#48c'])
ax.set_title(f'Home-team win rate = {dist[1]:.1%}')
ax.set_ylabel('proportion'); ax.set_xlabel('')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

HOME_TEAM_WINS
客隊勝    0.440289
主隊勝    0.559711
Name: proportion, dtype: float64


C:\Users\user\AppData\Local\Temp\ipykernel_42492\701930916.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=0); plt.tight_layout(); plt.show()


主場勝率約 56%，符合 NBA 歷史（55–60%）。這也是 baseline「主場必勝」的準確率下限，
模型必須明顯超過它。

## 3. 各賽季比賽數與主場勝率

In [4]:
by_season = games.groupby('SEASON').agg(
    games=('HOME_TEAM_WINS', 'size'),
    home_win_rate=('HOME_TEAM_WINS', 'mean'),
)
display(by_season)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_season['games'].plot(kind='bar', ax=axes[0], color='#48c')
axes[0].set_title('Games per season'); axes[0].set_xlabel('season')
by_season['home_win_rate'].plot(marker='o', ax=axes[1], color='#c44')
axes[1].axhline(0.5, ls='--', c='gray'); axes[1].set_title('Home win rate by season')
axes[1].set_ylim(0.45, 0.65)
plt.tight_layout(); plt.show()

,games,home_win_rate
SEASON,,
2018,1230,0.592683
2019,1059,0.551464
2020,1080,0.543519
2021,1230,0.543902
2022,1230,0.580488
2023,1230,0.543089


C:\Users\user\AppData\Local\Temp\ipykernel_42492\4262119695.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


> 註：2019（2019-20）與 2020（2020-21）賽季因疫情場次較少，屬正常現象。

## 4. 缺值檢查

In [5]:
na = games.isna().mean().sort_values(ascending=False)
print('原始 games 缺值比例（前 10）：')
print((na[na > 0] * 100).round(2).astype(str) + ' %' if (na > 0).any() else '無缺值')

原始 games 缺值比例（前 10）：
FT_PCT_away    0.01 %
dtype: str


## 5. 得分分布（主 vs 客）

In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(games['PTS_home'], color='#48c', label='Home PTS', kde=True, stat='density', alpha=0.5)
sns.histplot(games['PTS_away'], color='#c44', label='Away PTS', kde=True, stat='density', alpha=0.5)
ax.legend(); ax.set_title('Points scored: home vs away'); ax.set_xlabel('points')
plt.tight_layout(); plt.show()
print(f"主隊平均得分 {games.PTS_home.mean():.1f}，客隊平均得分 {games.PTS_away.mean():.1f}")

主隊平均得分 113.5，客隊平均得分 111.4


C:\Users\user\AppData\Local\Temp\ipykernel_42492\3228725106.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. 賽前特徵與標籤的關係

In [7]:
feats = features.build_features(games)
feat_cols = features.feature_columns(feats)
print(f'特徵表：{feats.shape[0]} 列 × {len(feat_cols)} 個特徵')

# 各特徵與標籤的相關係數（絕對值前 12）
corr = feats[feat_cols + [config.LABEL]].corr()[config.LABEL].drop(config.LABEL)
top = corr.reindex(corr.abs().sort_values(ascending=False).index).head(12)
ax = top.plot(kind='barh', figsize=(7, 5), color=np.where(top > 0, '#48c', '#c44'))
ax.set_title('Correlation with HOME_TEAM_WINS (top 12)'); ax.invert_yaxis()
plt.tight_layout(); plt.show()
top

特徵表：7059 列 × 30 個特徵


C:\Users\user\AppData\Local\Temp\ipykernel_42492\3154070743.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


diff_season_winrate        0.271569
diff_venue_winrate         0.254823
diff_winrate_last10        0.249828
home_season_winrate        0.219884
home_venue_winrate         0.211408
home_winrate_last10        0.208385
diff_winrate_last5         0.199071
diff_pts_for_last10        0.166616
home_winrate_last5         0.165892
away_season_winrate       -0.164715
diff_pts_against_last10   -0.163900
away_winrate_last10       -0.146914
Name: HOME_TEAM_WINS, dtype: float64

## 7. 差值特徵的預測力（示意）

In [8]:
col = 'diff_season_winrate'
d = feats.dropna(subset=[col]).copy()
d['bin'] = pd.qcut(d[col], 10, duplicates='drop')
rate = d.groupby('bin', observed=True)[config.LABEL].mean()
ax = rate.plot(marker='o', figsize=(8, 4))
ax.axhline(0.5, ls='--', c='gray')
ax.set_title('主隊勝率 vs (主-客) 賽季勝率差 分位'); ax.set_ylabel('home win rate')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_42492\2674136885.py:8: UserWarning: Glyph 20027 (\N{CJK UNIFIED IDEOGRAPH-4E3B}) missing from font(s) Arial.
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_42492\2674136885.py:8: UserWarning: Glyph 38538 (\N{CJK UNIFIED IDEOGRAPH-968A}) missing from font(s) Arial.
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_42492\2674136885.py:8: UserWarning: Glyph 21213 (\N{CJK UNIFIED IDEOGRAPH-52DD}) missing from font(s) Arial.
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_42492\2674136885.py:8: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from font(s) Arial.
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
C:\Users\user\AppData\Local\Temp\ipykernel_42492\2674136885.py:8: UserWarning: Glyph 23458 (\N{CJK UNIFIED IDEOG

**小結**
- 主場勝率約 56%，是模型必須超越的下限。
- 差值型特徵（主-客的近期/賽季勝率）與主隊勝負呈明顯單調關係，具預測力。
- 滾動特徵只用「該場之前」的資料計算（防洩漏），季初少數樣本為 NaN 屬正常，建模時丟棄暖機期。
- 下一步：baseline 對照 + Logistic Regression（見 `python -m src.baseline`），開發③ 再加 Elo 與 XGBoost。